<a href="https://colab.research.google.com/github/gcs0/PythonModellingAA/blob/main/Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import shutil
import os
# Download latest version
path = kagglehub.dataset_download("jhbros/enron-email-dataset")

shutil.copytree(path, "/content/drive/MyDrive/AASetFiles/Enron", dirs_exist_ok=True)


100%|██████████| 319M/319M [00:01<00:00, 171MB/s]

Extracting files...


'/content/drive/MyDrive/AASetFiles/Enron'

In [ ]:
from pypdf import PdfReader
import re

# Read the PDF file
pdf_path = '/content/2025.ranlp-1.123.pdf'
reader = PdfReader(pdf_path)

full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

print(f"Total pages extracted: {len(reader.pages)}")
print("\n--- Abstract / Introduction (First 1500 chars) ---")
print(full_text[:1500])

print("\n--- Mentions of 'final model' or 'architecture' ---")
# Search for context around 'final model' or related keywords to understand the implementation
keywords = r'.{0,300}(final model|architecture|authorship attribution).{0,300}'
matches = re.finditer(keywords, full_text, re.IGNORECASE | re.DOTALL)

count = 0
for match in matches:
    # Print first few matches to gather context
    if count < 10:
        # Clean up newlines for readable output
        clean_match = match.group(0).replace('\n', ' ')
        print(f"\nMatch {count+1}: ...{clean_match}...")
        count += 1

Total pages extracted: 8

--- Abstract / Introduction (First 1500 chars) ---
Proceedings of Recent Advances in Natural Language Processing ,pages 1066–1073
V arna, Sep 8–10, 2025
https://doi.org/10.26615/978-954-452-098-4-123
1066
Modelling the Relative Contributions of Stylistic Features in Forensic
Authorship Attribution
G. C ¸ a˘gatay Sat and John Blake and Evgeny Pyshkin
School of Computer Science and Engineering
University of Aizu
Aizuwakamatsu
Japan
s1312006,jblake,pyshe@u-aizu.ac.jp
Abstract
This paper explores the extent to which stylis-
tic features contribute to the task of authorship
attribution in forensic contexts. Drawing on a
filtered subset of the Enron email corpus, the
study operationalizes stylistic indicators across
four groups: lexical, syntactic, orthographic,
and discoursal. Using R Programming Lan-
guage for feature engineering and logistic re-
gression modelling, we systematically assessed
both the individual and interactive effects of
these features on attribu

In [ ]:
import os
import email
import numpy as np
import pandas as pd
import spacy
from collections import Counter
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

# Load English tokenizer, tagger, parser and NER
try:
    nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])
except:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

# ==========================================
# 1. Load Data and Extract Features
# ==========================================
def parse_enron_email(message):
    msg = email.message_from_string(message)
    return msg.get('From'), msg.get_payload()

# Load a subset of the dataset (Enron is very large)
dataset_file = os.path.join(path, "emails.csv")
print(f"Loading data from {dataset_file}...")
# Reading a subset for performance. Increase nrows for better results.
df_raw = pd.read_csv(dataset_file, nrows=5000)

# Extract author and body
df_raw['Author'], df_raw['Body'] = zip(*df_raw['message'].map(parse_enron_email))
df_raw = df_raw.dropna(subset=['Author', 'Body'])

# Select top 5 authors for the classification task to balance speed and representation
top_authors = df_raw['Author'].value_counts().head(5).index
df = df_raw[df_raw['Author'].isin(top_authors)].copy()
print(f"Selected {len(df)} emails from top 5 authors.")

def extract_stylistic_features(texts):
    features_list = []
    for doc in nlp.pipe(texts, batch_size=50):
        total_tokens = len(doc)
        if total_tokens == 0:
            total_tokens = 1 # Prevent division by zero

        pos_counts = Counter([token.pos_ for token in doc])

        features = {
            'RelADJ': pos_counts.get('ADJ', 0) / total_tokens,
            'RelDET': pos_counts.get('DET', 0) / total_tokens,
            'RelNOUN': pos_counts.get('NOUN', 0) / total_tokens,
            'RelPRON': pos_counts.get('PRON', 0) / total_tokens,
            'RelPROPN': pos_counts.get('PROPN', 0) / total_tokens,
            'RelPUNCT': pos_counts.get('PUNCT', 0) / total_tokens,
            'RelSPACE': pos_counts.get('SPACE', 0) / total_tokens,
            # Simple approximation for CTTR (Corrected Type-Token Ratio)
            'CTTR': len(set([t.text.lower() for t in doc])) / np.sqrt(2 * total_tokens),
            # ngram_sim requires complex author-level profiling, using random as placeholder for structure
            'ngram_sim': np.random.rand()
        }
        features_list.append(features)
    return pd.DataFrame(features_list)

print("Extracting linguistic features (this may take a minute)...")
X_base = extract_stylistic_features(df['Body'].astype(str).tolist())
y = df['Author'].astype('category').cat.codes.values

# ==========================================
# 2. Creating Interaction Terms (Final Model)
# ==========================================
def add_interaction_terms(df_features):
    features_to_interact = ['RelADJ', 'RelDET', 'RelNOUN', 'RelPRON',
                            'RelPROPN', 'RelPUNCT', 'RelSPACE', 'CTTR']

    for feature in features_to_interact:
        df_features[f'ngram_sim_x_{feature}'] = df_features['ngram_sim'] * df_features[feature]
    return df_features

X_final = add_interaction_terms(X_base)

# Train, validation, and test splits
X_temp, X_test, y_temp, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

# Scale features (critical for Elastic Net)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 3. Model Training (Elastic Net)
# ==========================================
# SGDClassifier with 'elasticnet' penalty for log loss (logistic regression)
model = SGDClassifier(loss='log_loss', penalty='elasticnet', max_iter=2000, random_state=42)

param_grid = {
    'alpha': [1e-4, 1e-3, 1e-2, 1e-1],
    'l1_ratio': [0.15, 0.5, 0.85]
}

print("Tuning Elastic Net hyperparameters...")
grid_search = GridSearchCV(model, param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_train_scaled, y_train)

best_model = grid_search.best_estimator_
print(f"Optimal Parameters found: {grid_search.best_params_}")

# ==========================================
# 4. Final Evaluation
# ==========================================
y_pred = best_model.predict(X_test_scaled)

print("\n--- Final Model Evaluation on Test Set ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

### 1. Imports and Setup

In [ ]:
import os
import email
import numpy as np
import pandas as pd
import spacy
from collections import Counter
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

# Load English tokenizer, tagger, parser and NER
try:
    nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])
except:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

### 2. Load Data and Parse Emails

In [ ]:
!pip install textacy -q
import pandas as pd
import re
from textacy import preprocessing

# Reading a much larger subset to ensure a robust number of authors and samples.
df_raw = pd.read_csv("/content/drive/MyDrive/AASetFiles/Enron/enron.csv", nrows=200000)

# The dataset already has 'From' and 'Message' columns.
text_column = 'Message'
author_column = 'From'

if text_column not in df_raw.columns or author_column not in df_raw.columns:
    raise KeyError(f"Make sure '{text_column}' and '{author_column}' exist in the dataset.")

# Assign Author and Body
df_raw['Author'] = df_raw[author_column]
df_raw['Body'] = df_raw[text_column]

# Clean out forwarded emails and replies using the Subject column
if 'Subject' in df_raw.columns:
    # Filter out subjects starting with RE:, FW:, FWD: (case insensitive)
    df_raw = df_raw[~df_raw['Subject'].astype(str).str.match(r'^\s*(re|fw|fwd)\s*:', case=False, na=False)]

df_raw = df_raw.dropna(subset=['Author', 'Body'])

# --- Better Text Filtering ---
def clean_enron_body(text):
    text = str(text)
    # Lowercase text (remove capitalization)
    text = text.lower()

    # Remove ONLY URLs using textacy
    text = preprocessing.replace.urls(text, repl="")

    # Remove 'Original Message' or 'Forwarded by' blocks and everything after
    text = re.sub(r'-+original message-+.*', '', text, flags=re.DOTALL)
    text = re.sub(r'-+forwarded by-+.*', '', text, flags=re.DOTALL)
    # Remove email metadata headers like To:, From:, Sent:, cc:, bcc:, Subject: if they appear in body
    text = re.sub(r'^(to|from|sent|cc|bcc|subject):\s.*$', '', text, flags=re.MULTILINE)

    # Remove multiple whitespaces/newlines
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_raw['Body'] = df_raw['Body'].apply(clean_enron_body)

# --- EXACT MIRROR OF R SCRIPT: split_data.R filter_authors_by_count ---
author_counts = df_raw['Author'].value_counts()
# Exactly 20 to 30 documents per author
valid_authors = author_counts[(author_counts >= 20) & (author_counts <= 30)].index
df = df_raw[df_raw['Author'].isin(valid_authors)].reset_index(drop=True)

print(f"Selected {len(df)} emails from {len(df['Author'].unique())} authors (strictly 20-30 docs per author).")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.6/321.6 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 63.8 MB/s eta 0:00:00
Selected 6739 emails from 274 authors (strictly 20-30 docs per author).


### 3. Extract Stylistic Features

In [24]:
def extract_stylistic_features(texts):
    features_list = []
    for doc in nlp.pipe(texts, batch_size=50):
        # R: remove_punct = TRUE, remove_numbers = TRUE, remove_url = TRUE for CTTR
        valid_tokens = [t.text.lower() for t in doc if not t.is_punct and not t.like_num and not t.like_url]
        total_valid = max(len(valid_tokens), 1)
        types = len(set(valid_tokens))
        cttr = types / np.sqrt(2 * total_valid)

        # R: remove_punct = FALSE, remove_numbers = TRUE, remove_url = TRUE for Base Tokens
        base_tokens = [t for t in doc if not t.like_num and not t.like_url]
        total_tokens = max(len(base_tokens), 1)

        alpha_toks = [t.text for t in doc if t.is_alpha]
        awl = np.mean([len(w) for w in alpha_toks]) if alpha_toks else 0

        pos_counts = Counter([token.pos_ for token in base_tokens])

        features = {
            'Tokens': total_tokens,
            'CTTR': min(cttr, 12.0), # pmin(..., 12) from R code
            'AWL': awl
        }

        pos_tags = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SPACE', 'VERB']
        for pos in pos_tags:
            features[pos] = pos_counts.get(pos, 0)
            features[f'Rel{pos}'] = features[pos] / total_tokens

        # Specific punctuation frequency counts used in the R model's interactions
        squote_count = sum(1 for t in doc if "'" in t.text)
        dash_count = sum(1 for t in doc if "-" in t.text or "—" in t.text)
        features['RelSQuote'] = squote_count / total_tokens
        features['RelDash'] = dash_count / total_tokens

        features_list.append(features)
    return pd.DataFrame(features_list)

print("Extracting raw and relative linguistic features...")
df_features = extract_stylistic_features(df['Body'].astype(str).tolist())
df_combined = pd.concat([df.reset_index(drop=True), df_features], axis=1)

Extracting raw and relative linguistic features...


### 4. Create Interaction Terms & Split Data

In [25]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import random
import numpy as np
import math

# ==== 1. EXACT AUTHOR SPLIT FROM split_data.R ====
seed = 1938
np.random.seed(seed)
random.seed(seed)

authors = df_combined['Author'].unique()
shuffled_authors = list(authors)
random.shuffle(shuffled_authors)

total_size = math.floor(0.95 * len(shuffled_authors))
train_size = math.floor(0.7 * total_size)

test_size = 5
valid_size = total_size - train_size - test_size

train_authors = shuffled_authors[:train_size]
valid_authors = shuffled_authors[train_size:train_size + valid_size]
# Select a new set of 5 authors for the test set, ensuring they are not in train_authors or valid_authors
remaining_authors = [author for author in shuffled_authors if author not in train_authors and author not in valid_authors]
test_authors = random.sample(remaining_authors, test_size)

train_df = df_combined[df_combined['Author'].isin(train_authors)].copy().reset_index(drop=True)
val_df = df_combined[df_combined['Author'].isin(valid_authors)].copy().reset_index(drop=True)
test_df = df_combined[df_combined['Author'].isin(test_authors)].copy().reset_index(drop=True)

print(f"Author Split: Train={len(train_authors)}, Valid={len(valid_authors)}, Test={len(test_authors)}")
print(f"Document Split: Train={len(train_df)}, Valid={len(val_df)}, Test={len(test_df)}")

# ==== 2. BUILD PROFILES AND PAIRWISE SETS (LOO STRATEGY) ====
pos_tags = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SPACE', 'VERB']
stylistic_cols = [f'Rel{pos}' for pos in pos_tags] + ['CTTR', 'AWL', 'RelSQuote', 'RelDash']
# Use only the 9 features matched in the final R model
interaction_features = ['RelSQuote', 'RelDash', 'RelADJ', 'RelDET', 'RelNOUN', 'RelPRON', 'RelPROPN', 'RelPUNCT', 'RelSPACE']

def get_style_profile(df_subset):
    # Mirror R's create_author_level_features: taking the mean of document-level features
    if len(df_subset) == 0:
        return "", {col: 0.0 for col in stylistic_cols}

    style = {col: df_subset[col].mean() for col in stylistic_cols}
    text = ' '.join(df_subset['Body'].astype(str))
    return text, style

def create_pairwise_dataset(data_df, count_vec=None, is_train=False):
    candidates_list = data_df['Author'].unique()

    standard_texts = {}
    standard_styles = {}
    for author in candidates_list:
        df_sub = data_df[data_df['Author'] == author]
        text, style = get_style_profile(df_sub)
        standard_texts[author] = text
        standard_styles[author] = style

    if is_train:
        # Mirroring R quanteda dfm: word 1-2 grams, raw counts, no TF-IDF
        count_vec = CountVectorizer(analyzer='word', ngram_range=(1, 2), min_df=5)
        count_vec.fit(data_df['Body'].astype(str).tolist())

    records = []
    doc_texts = data_df['Body'].astype(str).tolist()
    doc_vectors = count_vec.transform(doc_texts)

    cand_vectors_dict = {author: count_vec.transform([standard_texts[author]]) for author in candidates_list}

    for i, row in enumerate(data_df.itertuples()):
        true_author = row.Author

        df_loo = data_df[(data_df['Author'] == true_author) & (data_df.index != i)]
        loo_text, loo_style = get_style_profile(df_loo)
        loo_vector = count_vec.transform([loo_text])

        for author in candidates_list:
            if author == true_author:
                cand_style = loo_style
                cand_vec = loo_vector
            else:
                cand_style = standard_styles[author]
                cand_vec = cand_vectors_dict[author]

            # Prevent 0-division in cosine sim if vector is empty
            doc_v = doc_vectors[i]
            if doc_v.nnz == 0 or cand_vec.nnz == 0:
                ngram_sim = 0.0
            else:
                ngram_sim = cosine_similarity(doc_v, cand_vec)[0, 0]

            record = {}
            for col in stylistic_cols:
                record[col] = abs(getattr(row, col) - cand_style[col])
            record['ngram_sim'] = ngram_sim

            for feat in interaction_features:
                record[f'ngram_sim_x_{feat}'] = ngram_sim * record[feat]

            record['Label'] = 1 if author == true_author else 0
            record['Doc_ID'] = i
            record['Candidate'] = author
            record['True_Author'] = true_author
            records.append(record)

    return pd.DataFrame(records), count_vec

print("Generating pairwise features for Train set (with LOO)...")
train_pairs, count_vectorizer = create_pairwise_dataset(train_df, is_train=True)

print("Generating pairwise features for Valid set (with LOO)...")
val_pairs, _ = create_pairwise_dataset(val_df, count_vec=count_vectorizer, is_train=False)

interaction_cols = [f'ngram_sim_x_{feat}' for feat in interaction_features]
# STRICTLY match the paper: only ngram_sim and interactions. No base stylistic features to avoid collinearity
feature_cols = ['ngram_sim'] + interaction_cols

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_pairs[feature_cols])
y_train = train_pairs['Label'].values

X_val_scaled = scaler.transform(val_pairs[feature_cols])
y_val = val_pairs['Label'].values

Author Split: Train=182, Valid=73, Test=5
Document Split: Train=4481, Valid=1775, Test=114
Generating pairwise features for Train set (with LOO)...
Generating pairwise features for Valid set (with LOO)...


### 5. Model Training (Elastic Net)

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

print("Training Binary Elastic Net classifier matching exact R configuration (l1_ratio=0.8)...")

# Using LogisticRegression with 'saga' solver which is mathematically closer
# to glmnet's coordinate descent than SGD.
model = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    l1_ratio=0.8,
    max_iter=5000, # Increased significantly to ensure coefficients converge
    random_state=42
)

# C is the inverse of regularization strength (1 / lambda).
# Expanding the grid to ensure we find the true optimal value.
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0, 100.0]
}

# Train on a representative subset to speed up CV
cv_size = min(len(X_train_scaled), 20000)
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='neg_log_loss', n_jobs=-1)
grid_search.fit(X_train_scaled[:cv_size], y_train[:cv_size])

best_model = grid_search.best_estimator_
print(f"Optimal C found: {best_model.C}")

# Retrain on the full training set with the best penalty parameter
print("Retraining on full dataset with optimal parameters...")
best_model.fit(X_train_scaled, y_train)
print("Training complete.")

Training Binary Elastic Net classifier matching exact R configuration (l1_ratio=0.8)...
Optimal C found: 0.1
Retraining on full dataset with optimal parameters...
Training complete.


### 6. Final Evaluation

In [27]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

print(f"Evaluating {len(test_df)} documents from the test set using LOO profiles...")

# 1. Generate pairwise test dataset using the exact same LOO strategy as train/val
test_pairs, _ = create_pairwise_dataset(test_df, count_vec=count_vectorizer, is_train=False)

# 2. Scale features using the scaler fitted on the training set
X_test_scaled_final = scaler.transform(test_pairs[feature_cols])

# 3. Predict probabilities for each test candidate
test_pairs['Probability'] = best_model.predict_proba(X_test_scaled_final)[:, 1]

predictions = []
truths = []

for doc_id, group in test_pairs.groupby('Doc_ID'):
    # Top-1 selection
    best_candidate = group.loc[group['Probability'].idxmax()]['Candidate']
    true_author = group.iloc[0]['True_Author']

    predictions.append(best_candidate)
    truths.append(true_author)

print(f"\n--- Final Model Evaluation on Test Set ---")
print(f"Top-1 Authorship Attribution Accuracy: {accuracy_score(truths, predictions):.4f}")

Evaluating 114 documents from the test set using LOO profiles...

--- Final Model Evaluation on Test Set ---
Top-1 Authorship Attribution Accuracy: 0.9123


### Model Coefficients

In [22]:
print("--- Model Coefficients (Un-scaled to match R's glmnet output) ---")

# Get coefficients for the positive class
raw_coefficients = best_model.coef_[0]

# R's glmnet standardizes internally but returns coefficients on the original scale.
# To match this, we must divide our scaled coefficients by the standard deviation of each feature.
unscaled_coefficients = raw_coefficients / scaler.scale_

# Intercept needs to be adjusted as well, but for relative feature importance,
# the coefficients themselves are the main focus.
intercept = best_model.intercept_[0] - sum(raw_coefficients * scaler.mean_ / scaler.scale_)
print(f"(Intercept) \t {intercept:.4f}\n")

coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': unscaled_coefficients
}).sort_values(by='Coefficient', ascending=False)

display(coef_df.round(4))

--- Model Coefficients (Un-scaled to match R's glmnet output) ---
(Intercept) 	 -9.3479



,Feature,Coefficient
2,RelDET,17.1929
0,ngram_sim,12.3415
4,RelPRON,9.9819
3,RelNOUN,5.8506
1,RelADJ,4.7705
13,ngram_sim_x_RelPROPN,3.0573
5,RelPROPN,2.3620
6,RelPUNCT,2.1831
7,RelSPACE,0.0000
15,ngram_sim_x_RelSPACE,0.0000


In [ ]:
import os

repo_path = "/content/ModellingAA"
files_to_check = [
    "features.R",
    "create_author_level_features.R",
    "create_ngram_similarity.R"
]

for file_name in files_to_check:
    filepath = os.path.join(repo_path, file_name)
    if os.path.exists(filepath):
        print(f"========== {file_name} ==========")
        with open(filepath, 'r') as f:
            # Print the entire file to accurately analyze feature generation
            print(f.read())
        print("\n")
    else:
        print(f"{file_name} not found in {repo_path}")

========== features.R ==========
# 02_features.R

# Source punctuation functions
source("puncmarkFunc.R")

compute_base_features <- function(corpus, Df) {
  Df$Alphabetic <- str_count(corpus, "[A-Za-z]")
  Df$AlphaToken <- ntoken(tokens(corpus, remove_punct = TRUE, remove_numbers = TRUE, remove_url = TRUE))
  Df$Token <- ntoken(tokens(corpus, remove_punct = FALSE, remove_numbers = TRUE, remove_url = TRUE))
  Df$Uppercase <- str_count(corpus, "[A-Z]")
  Df$RelCase <- ifelse(Df$Alphabetic == 0, 0, Df$Uppercase / Df$Alphabetic)
  
  
  cttr_result <- textstat_lexdiv(tokens(corpus, remove_punct = TRUE, remove_numbers = TRUE, remove_url = TRUE), measure = "CTTR")
  Df$CTTR <- pmin(cttr_result$CTTR, 12)
  Df<- add_punct_counts(corpus, Df)
  Df<- add_punct_rel_freq(Df)
  return(Df)
}

add_pos_scaled_features <- function(corpus, df) {
  # Build a stable doc_id vector aligned to df rows
  doc_id_all <- if (inherits(corpus, "corpus")) {
    quanteda::docnames(corpus)
  } else if (!is.null(names(

In [ ]:
import os

files_to_check = [
    "ngram_matrix_calc.R",
    "create_ngram_similarity.R"
]

for file_name in files_to_check:
    filepath = os.path.join(repo_path, file_name)
    if os.path.exists(filepath):
        print(f"\n{'='*20} {file_name} {'='*20}")
        with open(filepath, 'r') as f:
            # Print the entire file to accurately analyze feature generation
            print(f.read())
    else:
        print(f"\n{file_name} not found in {repo_path}")


==================== ngram_matrix_calc.R ====================
prepare_doc_author_comparison <- function(doc_df_name,
                                          author_df_name,
                                          ngram_df_name,
                                          feature_cols_name,
                                          doc_id_col = "doc_id",
                                          doc_author_col = "Author",
                                          author_id_col = "Author",
                                          output_name = NULL,
                                          envir = .GlobalEnv) {
  # Fetch objects from their names
  doc_df <- get(doc_df_name, envir = envir)
  author_df <- get(author_df_name, envir = envir)
  ngram_df <- get(ngram_df_name, envir = envir)
  feature_cols <- get(feature_cols_name, envir = envir)
  
  # Run comparison using previously defined function
  result_df <- compare_doc_to_authors(
    doc_df = doc_df,
    author_df = author_df,
  

In [ ]:
import re

filepath = '/content/ModellingAA/create_ngram_similarity.R'
with open(filepath, 'r') as f:
    content = f.read()

    # Search for tokenization, n-gram, and dfm creation steps
    print("--- Extracted text processing logic from create_ngram_similarity.R ---")
    lines = content.split('\n')
    for i, line in enumerate(lines):
        if any(keyword in line for keyword in ['tokens', 'dfm', 'weight', 'tfidf', 'ngram', 'ngrams']):
            # Print a window around the match to get context
            print(f"Line {i+1}: {line.strip()}")

--- Extracted text processing logic from create_ngram_similarity.R ---
Line 11: #' @param ngram_range Integer vector specifying n-gram range (default: 1:2)
Line 12: #' @param min_termfreq Minimum term frequency for dfm_trim (default: 5)
Line 24: #' ngram_matrix <- create_ngram_similarity_matrix(
Line 32: #' ngram_matrix <- create_ngram_similarity_matrix(
Line 35: #'   ngram_range = 1:3,
Line 42: create_ngram_similarity_matrix <- function(corpus,
Line 46: ngram_range = 1:2,
Line 93: if (!is.numeric(ngram_range) || length(ngram_range) != 2) {
Line 94: stop("ngram_range must be a numeric vector of length 2 (e.g., 1:2)")
Line 104: # Prepare tokens and dfm
Line 105: message("Preparing n-gram tokens and document-feature matrix...")
Line 106: toks <- tokens(corpus, remove_punct = remove_punct, remove_numbers = remove_numbers) %>%
Line 107: tokens_ngrams(n = ngram_range)
Line 109: dfm_all <- dfm(toks) %>%
Line 110: dfm_trim(min_termfreq = min_termfreq)
Line 125: doc_mat <- as(dfm_all, "dgCMatr